In [7]:
import os
import re
import random
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from tqdm import tqdm


project_dir = Path.cwd()
for p in [project_dir, *project_dir.parents]:
    if (p / "SYMBA-GSoC2026" / "src").exists():
        sys.path.insert(0, str(p / "SYMBA-GSoC2026"))
        sys.path.insert(0, str(p / "SYMBA-GSoC2026" / "src"))
        break

from src.data.tokeniser import TargetTokenizer

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
class DataConfig:
    MAX_LENGTH = 150
    NODE_FEATURE_DIM = 64
    EDGE_FEATURE_DIM = 32

config = DataConfig()

def parse_txt_file(filepath: Path) -> list:
    """Parse one raw .txt file and return list of row dicts."""
    rows = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        content = f.read()

    for line in content.splitlines():
        line = line.strip()
        if not line.startswith("Interaction:"):
            continue

       
        body = line[len("Interaction:"):].strip()
        body_norm = re.sub(r'\s+:\s+', ' : ', body)
        parts = body_norm.rsplit(' : ', 3)

        if len(parts) != 4:
            continue

        interaction, topology, amplitude, squared_amplitude = [p.strip() for p in parts]
        rows.append({
            "interaction": interaction,
            "topology": topology,
            "amplitude": amplitude,
            "squared_amplitude": squared_amplitude,
        })
    return rows

def load_all_raw_data(raw_dirs: list[Path]) -> pd.DataFrame:
    all_rows = []
    for raw_dir in raw_dirs:
        txt_files = sorted(raw_dir.glob("*.txt"))
        print(f"Found {len(txt_files)} files in {raw_dir.name}")
        for fp in txt_files:
            rows = parse_txt_file(fp)
            all_rows.extend(rows)
            print(f"  {fp.name:<55} -> {len(rows):4d} records")

    df = pd.DataFrame(all_rows, columns=["interaction", "topology", "amplitude", "squared_amplitude"])
    print(f"\n[OK] Combined rows: {len(df)}")
    return df

def split_dataframe(
    df: pd.DataFrame,
    ratios: tuple[float, float, float] = (0.8, 0.1, 0.1),
    seed: int = 42,
    shuffle: bool = True,
):
    assert abs(sum(ratios) - 1.0) < 1e-9, "Split ratios must sum to 1."
    if shuffle:
        df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    n = len(df)
    n_train = int(n * ratios[0])
    n_val = int(n * ratios[1])

    train = df.iloc[:n_train].reset_index(drop=True)
    val = df.iloc[n_train : n_train + n_val].reset_index(drop=True)
    test = df.iloc[n_train + n_val :].reset_index(drop=True)

    print(f"Split sizes -> train: {len(train)}, val: {len(val)}, test: {len(test)}")
    return train, val, test

def save_splits(train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    train.to_csv(output_dir / "train.csv", index=False)
    val.to_csv(output_dir / "val.csv", index=False)
    test.to_csv(output_dir / "test.csv", index=False)
    print(f"Saved CSV splits to: {output_dir}")

In [9]:
workspace_root = Path.cwd()
for p in [workspace_root, *workspace_root.parents]:
    if (p / "SYMBA-GSoC2026").exists():
        workspace_root = p
        break

project_root = workspace_root / "SYMBA-GSoC2026"
raw_dirs = [project_root / "QED data", project_root / "QCD data"]
preprocessed_dir = project_root / "preprocessed"

raw_df = load_all_raw_data(raw_dirs)
train_df, val_df, test_df = split_dataframe(raw_df, ratios=(0.8, 0.1, 0.1), seed=42, shuffle=True)
save_splits(train_df, val_df, test_df, preprocessed_dir)

print("\nPhase 1 complete.")
print(f"train.csv: {len(train_df)} rows")
print(f"val.csv:   {len(val_df)} rows")
print(f"test.csv:  {len(test_df)} rows")

Found 10 files in QED data
  QED-2-to-2-diag-TreeLevel-0.txt                         ->   54 records
  QED-2-to-2-diag-TreeLevel-1.txt                         ->   50 records
  QED-2-to-2-diag-TreeLevel-2.txt                         ->   46 records
  QED-2-to-2-diag-TreeLevel-3.txt                         ->   42 records
  QED-2-to-2-diag-TreeLevel-4.txt                         ->   38 records
  QED-2-to-2-diag-TreeLevel-5.txt                         ->   34 records
  QED-2-to-2-diag-TreeLevel-6.txt                         ->   30 records
  QED-2-to-2-diag-TreeLevel-7.txt                         ->   26 records
  QED-2-to-2-diag-TreeLevel-8.txt                         ->   22 records
  QED-2-to-2-diag-TreeLevel-9.txt                         ->   18 records
Found 7 files in QCD data
  QCD-2-to-2-diag-TreeLevel-0.txt                         ->   46 records
  QCD-2-to-2-diag-TreeLevel-1.txt                         ->   42 records
  QCD-2-to-2-diag-TreeLevel-2.txt                         -

In [10]:
from src.data.topology_parser import topology_to_pyg
def process_csv_to_graphs(csv_path: Path, config: DataConfig, tgt_vocab, tokenizer):
    print(f"Loading CSV: {csv_path}")
    df = pd.read_csv(csv_path)

    processed_graphs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Building {csv_path.stem} tensors"):
        graph_data = topology_to_pyg(row["topology"], config)

        raw_target = str(row["squared_amplitude"])
        tokens = tokenizer.tgt_tokenize(raw_target)
        
        
        token_ids = [tgt_vocab["<bos>"]] + [tgt_vocab.lookup_indices([t])[0] if t in tgt_vocab else tgt_vocab["<unk>"] for t in tokens] + [tgt_vocab["<eos>"]]
        
        padded_ids = token_ids[: config.MAX_LENGTH] + [tgt_vocab["<pad>"]] * max(0, config.MAX_LENGTH - len(token_ids))

        graph_data.y = torch.tensor(padded_ids, dtype=torch.long)
        processed_graphs.append(graph_data)

    return processed_graphs

In [ ]:
train_csv_path = preprocessed_dir / "train.csv"
print("--- Building Global Target Vocabulary ---")
train_df = pd.read_csv(train_csv_path)
special_symbols = ["<unk>", "<pad>", "<bos>", "<eos>"]
tokenizer = TargetTokenizer(train_df, special_symbols, UNK_IDX=0)
global_vocab = tokenizer.build_tgt_vocab()

splits = ["train", "val", "test"]

for split in splits:
    csv_path = preprocessed_dir / f"{split}.csv"
    output_pt = preprocessed_dir / f"{split}_graphs.pt"

    if not csv_path.exists():
        print(f"Skipping missing file: {csv_path}")
        continue

    
    print(f"Processing {split} split...")
    graph_dataset = process_csv_to_graphs(csv_path, config, global_vocab, tokenizer)
    
    
    torch.save({
        'dataset': graph_dataset,
        'vocab': global_vocab 
    }, output_pt)
    
    print(f" Saved {len(graph_dataset)} PyG graphs to {output_pt.name}\n")

print(" All data splits are ready")

NameError: name 'preprocessed_dir' is not defined

: 

In [ ]:
output_pt_path = preprocessed_dir / "train_graphs.pt"
if not output_pt_path.exists():
    raise FileNotFoundError(f"Expected file not found: {output_pt_path}")

saved_data = torch.load(output_pt_path, weights_only=False)
saved_dataset = saved_data["dataset"]
saved_vocab = saved_data["vocab"]

print("--- Verification ---")
print(f"Loaded file:   {output_pt_path.name}")
print(f"Total graphs: {len(saved_dataset)}")
print(f"Vocab size:   {len(saved_vocab)}")

sample_graph = saved_dataset[0]
print("\nSample graph [0]:")
print(f"- Nodes:        {sample_graph.num_nodes}")
print(f"- Edges:        {sample_graph.num_edges}")
print(f"- x shape:      {tuple(sample_graph.x.shape)}")
print(f"- edge_attr:    {tuple(sample_graph.edge_attr.shape)}")
print(f"- y shape:      {tuple(sample_graph.y.shape)}")

if hasattr(saved_vocab, "lookup_token"):
    decoded_target = [saved_vocab.lookup_token(int(idx)) for idx in sample_graph.y[:10]]
else:
    decoded_target = [saved_vocab.itos[int(idx)] for idx in sample_graph.y[:10]]
print(f"- Decoded y[:10]: {decoded_target}")

--- Verification ---
Total graphs: 475
Vocab size:   98

Sample graph [0]:
- Nodes:        6
- Edges:        18
- x shape:      (6, 64)
- edge_attr:    (18, 32)
- y shape:      (150,)
- Decoded y[:10]: ['<bos>', '1/9', '*', 'e', '^', '4', '*', '(', '16', '*']
